# 🔍 Database Validation & Health Check

**Comprehensive testing notebook to validate database state before API consumption.**

This notebook checks:
1. **league.db** - Rosters, matchups, NFL players
2. **projections.db** - Projections, player stats, team lineups
3. **montecarlo.db** - Simulation data
4. **odds.db** - Betting odds data
5. **Betting API** - Validates data matches what the Betting page expects

## Test Results Legend:
- ✅ **PASS** - Data is valid and ready for API consumption
- ⚠️ **WARNING** - Data exists but may have issues
- ❌ **FAIL** - Data is missing or invalid, needs attention

---

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Tuple, Optional

# ==================== CONFIGURATION ====================
CURRENT_WEEK = 13  # ⬅️ UPDATE THIS TO TEST SPECIFIC WEEK
EXPECTED_TEAMS = 12  # Number of fantasy teams in league
EXPECTED_MATCHUPS_PER_WEEK = 12  # 6 games × 2 teams
EXPECTED_SIMS_PER_RUN = 50000  # Monte Carlo simulations per run

# Database paths
NOTEBOOK_DIR = Path().absolute()
BACKEND_DIR = NOTEBOOK_DIR.parent
DB_DIR = BACKEND_DIR / "data" / "databases"

DB_LEAGUE = str(DB_DIR / "league.db")
DB_PROJECTIONS = str(DB_DIR / "projections.db")
DB_MONTECARLO = str(DB_DIR / "montecarlo.db")
DB_ODDS = str(DB_DIR / "odds.db")

# Expected valid values
VALID_POSITIONS = {'QB', 'RB', 'WR', 'TE', 'K', 'DST'}
INVALID_TEAMS = {'WSH', 'JAC', 'LA'}  # Should be standardized
EXPECTED_SOURCES = {'espn.com', 'fanduel.com', 'fantasypros.com', 'firstdown.studio', 'sleeper.com'}

# Test results storage
test_results = []

def record_test(category: str, test_name: str, status: str, message: str, details: str = ""):
    test_results.append({'category': category, 'test': test_name, 'status': status, 'message': message, 'details': details})
    icon = '✅' if status == 'PASS' else '⚠️' if status == 'WARNING' else '❌'
    print(f"{icon} [{status}] {test_name}: {message}")
    if details:
        print(f"   └─ {details}")

def get_week_string(week: int) -> str:
    return f"Week {week}"

print("✅ Setup complete!")
print(f"📅 Testing Week {CURRENT_WEEK}")
print(f"📁 Database directory: {DB_DIR}")
print("\n" + "="*70)

✅ Setup complete!
📅 Testing Week 13
📁 Database directory: /Users/samerfaizi/Documents/Code/TNC-Model-2025/backend/data/databases



---
## 1️⃣ League Database Tests (`league.db`)

In [2]:
print("="*70)
print("🏈 LEAGUE DATABASE TESTS (league.db)")
print("="*70 + "\n")

try:
    conn = sqlite3.connect(DB_LEAGUE)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    
    cursor.execute("SELECT COUNT(*) FROM rosters")
    roster_count = cursor.fetchone()[0]
    if roster_count >= EXPECTED_TEAMS:
        record_test("league.db", "Rosters exist", "PASS", f"Found {roster_count} rosters")
    else:
        record_test("league.db", "Rosters exist", "FAIL", f"Only {roster_count} rosters")
    
    cursor.execute("SELECT COUNT(*) FROM matchups WHERE week = ?", (CURRENT_WEEK,))
    matchup_count = cursor.fetchone()[0]
    if matchup_count == EXPECTED_MATCHUPS_PER_WEEK:
        record_test("league.db", f"Week {CURRENT_WEEK} matchups", "PASS", f"Found {matchup_count} matchups")
    elif matchup_count > 0:
        record_test("league.db", f"Week {CURRENT_WEEK} matchups", "WARNING", f"Found {matchup_count} (expected {EXPECTED_MATCHUPS_PER_WEEK})")
    else:
        record_test("league.db", f"Week {CURRENT_WEEK} matchups", "FAIL", "No matchups found")
    
    cursor.execute("SELECT matchup_id_number, COUNT(*) as cnt FROM matchups WHERE week = ? GROUP BY matchup_id_number", (CURRENT_WEEK,))
    pairs = cursor.fetchall()
    invalid = [p for p in pairs if p['cnt'] != 2]
    if len(pairs) == 6 and not invalid:
        record_test("league.db", "Matchup pairing", "PASS", "6 matchups, each with 2 teams")
    elif pairs:
        record_test("league.db", "Matchup pairing", "WARNING", f"{len(pairs)} matchup groups")
    
    cursor.execute("SELECT COUNT(*) FROM nfl_players")
    player_count = cursor.fetchone()[0]
    if player_count >= 1000:
        record_test("league.db", "NFL players loaded", "PASS", f"Found {player_count} players")
    else:
        record_test("league.db", "NFL players loaded", "FAIL", f"Only {player_count} players")
    
    cursor.execute("SELECT DISTINCT team FROM nfl_players WHERE team IS NOT NULL")
    teams = {row[0] for row in cursor.fetchall()}
    invalid_teams = teams & INVALID_TEAMS
    if not invalid_teams:
        record_test("league.db", "NFL player teams", "PASS", f"All {len(teams)} teams valid")
    else:
        record_test("league.db", "NFL player teams", "FAIL", f"Invalid: {invalid_teams}")
    
    conn.close()
except Exception as e:
    record_test("league.db", "Database connection", "FAIL", str(e))

🏈 LEAGUE DATABASE TESTS (league.db)

✅ [PASS] Rosters exist: Found 22 rosters
✅ [PASS] Week 13 matchups: Found 12 matchups
✅ [PASS] Matchup pairing: 6 matchups, each with 2 teams
✅ [PASS] NFL players loaded: Found 3968 players
✅ [PASS] NFL player teams: All 32 teams valid


---
## 2️⃣ Projections Database Tests (`projections.db`)

In [3]:
print("="*70)
print("📊 PROJECTIONS DATABASE TESTS")
print("="*70 + "\n")

week_str = get_week_string(CURRENT_WEEK)

try:
    conn = sqlite3.connect(DB_PROJECTIONS)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    
    cursor.execute("SELECT COUNT(*) FROM projections WHERE week = ?", (week_str,))
    proj_count = cursor.fetchone()[0]
    if proj_count >= 200:
        record_test("projections", f"Week {CURRENT_WEEK} projections", "PASS", f"Found {proj_count}")
    elif proj_count > 0:
        record_test("projections", f"Week {CURRENT_WEEK} projections", "WARNING", f"Only {proj_count}")
    else:
        record_test("projections", f"Week {CURRENT_WEEK} projections", "FAIL", "No projections")
    
    cursor.execute("SELECT source_website, COUNT(*) FROM projections WHERE week = ? GROUP BY source_website", (week_str,))
    sources = {row[0]: row[1] for row in cursor.fetchall()}
    missing = EXPECTED_SOURCES - set(sources.keys())
    if not missing:
        record_test("projections", "Source coverage", "PASS", f"All {len(EXPECTED_SOURCES)} sources")
    else:
        record_test("projections", "Source coverage", "WARNING", f"Missing: {missing}")
    print(f"   Sources: {sources}")
    
    cursor.execute("SELECT DISTINCT position FROM projections WHERE week = ?", (week_str,))
    positions = {row[0] for row in cursor.fetchall()}
    invalid_pos = positions - VALID_POSITIONS
    if not invalid_pos:
        record_test("projections", "Position standardization", "PASS", f"Valid: {sorted(positions)}")
    else:
        record_test("projections", "Position standardization", "FAIL", f"Invalid: {invalid_pos}")
    
    cursor.execute("SELECT DISTINCT team FROM projections WHERE week = ? AND team IS NOT NULL", (week_str,))
    teams = {row[0] for row in cursor.fetchall()}
    invalid_teams = teams & INVALID_TEAMS
    if not invalid_teams:
        record_test("projections", "Team standardization", "PASS", f"All {len(teams)} teams valid")
    else:
        record_test("projections", "Team standardization", "FAIL", f"Invalid: {invalid_teams}")
    
    cursor.execute("""
        SELECT source_website, player_first_name, player_last_name, position, COUNT(*) as cnt
        FROM projections WHERE week = ?
        GROUP BY source_website, player_first_name, player_last_name, position
        HAVING cnt > 1
    """, (week_str,))
    dupes = cursor.fetchall()
    if not dupes:
        record_test("projections", "No duplicates", "PASS", "No duplicate projections")
    else:
        record_test("projections", "No duplicates", "FAIL", f"{len(dupes)} duplicates found")
    
    conn.close()
except Exception as e:
    record_test("projections", "Database", "FAIL", str(e))

📊 PROJECTIONS DATABASE TESTS

✅ [PASS] Week 13 projections: Found 1578
⚠️ [WARNING] Source coverage: Missing: {'firstdown.studio'}
   Sources: {'espn.com': 246, 'fanduel.com': 546, 'fantasypros.com': 362, 'sleeper.com': 424}
✅ [PASS] Position standardization: Valid: ['DST', 'K', 'QB', 'RB', 'TE', 'WR']
✅ [PASS] Team standardization: All 33 teams valid
✅ [PASS] No duplicates: No duplicate projections


In [4]:
print("="*70)
print("🔗 SLEEPER MATCHING & PLAYER STATS")
print("="*70 + "\n")

week_str = get_week_string(CURRENT_WEEK)

try:
    conn = sqlite3.connect(DB_PROJECTIONS)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    
    cursor.execute("SELECT COUNT(*) FROM projections_with_sleeper WHERE week = ?", (week_str,))
    matched = cursor.fetchone()[0]
    cursor.execute("SELECT COUNT(*) FROM projections WHERE week = ?", (week_str,))
    total = cursor.fetchone()[0]
    if matched == total and matched > 0:
        record_test("sleeper_match", "Sleeper matching", "PASS", f"{matched}/{total} matched")
    elif matched > 0:
        record_test("sleeper_match", "Sleeper matching", "WARNING", f"{matched}/{total} matched ({matched/total*100:.1f}%)")
    else:
        record_test("sleeper_match", "Sleeper matching", "FAIL", "No matched projections")
    
    cursor.execute("SELECT COUNT(*) FROM player_week_stats WHERE week = ?", (CURRENT_WEEK,))
    stats = cursor.fetchone()[0]
    if stats >= 100:
        record_test("player_stats", f"Week {CURRENT_WEEK} stats", "PASS", f"{stats} player stats")
    elif stats > 0:
        record_test("player_stats", f"Week {CURRENT_WEEK} stats", "WARNING", f"Only {stats} stats")
    else:
        record_test("player_stats", f"Week {CURRENT_WEEK} stats", "FAIL", "No player stats")
    
    cursor.execute("SELECT COUNT(*) FROM player_week_stats WHERE week = ? AND (mu IS NULL OR sigma IS NULL OR mu < 0)", (CURRENT_WEEK,))
    invalid = cursor.fetchone()[0]
    if invalid == 0:
        record_test("player_stats", "Valid mu/sigma", "PASS", "All stats have valid values")
    else:
        record_test("player_stats", "Valid mu/sigma", "FAIL", f"{invalid} invalid records")
    
    cursor.execute("""
        SELECT position, COUNT(*) as cnt, AVG(mu) as avg_mu, AVG(sigma) as avg_sigma
        FROM player_week_stats WHERE week = ? GROUP BY position
    """, (CURRENT_WEEK,))
    print("   Position breakdown:")
    for row in cursor.fetchall():
        print(f"     {row['position']}: {row['cnt']} players, μ={row['avg_mu']:.1f}, σ={row['avg_sigma']:.2f}")
    
    conn.close()
except Exception as e:
    record_test("player_stats", "Error", "FAIL", str(e))

🔗 SLEEPER MATCHING & PLAYER STATS

✅ [PASS] Sleeper matching: 1578/1578 matched
✅ [PASS] Week 13 stats: 602 player stats
✅ [PASS] Valid mu/sigma: All stats have valid values
   Position breakdown:
     DST: 32 players, μ=5.7, σ=7.30
     K: 30 players, μ=7.8, σ=4.45
     QB: 81 players, μ=7.7, σ=8.08
     RB: 130 players, μ=5.6, σ=9.25
     TE: 118 players, μ=3.6, σ=8.24
     WR: 211 players, μ=4.4, σ=10.76


In [5]:
print("="*70)
print("👥 TEAM LINEUPS & SUMMARIES")
print("="*70 + "\n")

try:
    conn = sqlite3.connect(DB_PROJECTIONS)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    
    cursor.execute("SELECT COUNT(*) FROM team_lineups WHERE week = ?", (CURRENT_WEEK,))
    lineups = cursor.fetchone()[0]
    expected = EXPECTED_TEAMS * 9
    if lineups >= expected - 12:
        record_test("team_lineups", f"Week {CURRENT_WEEK} lineups", "PASS", f"{lineups} entries (~{expected} expected)")
    elif lineups > 0:
        record_test("team_lineups", f"Week {CURRENT_WEEK} lineups", "WARNING", f"{lineups}/{expected} entries")
    else:
        record_test("team_lineups", f"Week {CURRENT_WEEK} lineups", "FAIL", "No lineups")
    
    cursor.execute("SELECT COUNT(DISTINCT team_name) FROM team_lineups WHERE week = ?", (CURRENT_WEEK,))
    teams = cursor.fetchone()[0]
    if teams == EXPECTED_TEAMS:
        record_test("team_lineups", "All teams present", "PASS", f"{teams} teams")
    else:
        record_test("team_lineups", "All teams present", "WARNING", f"{teams}/{EXPECTED_TEAMS} teams")
    
    cursor.execute("SELECT COUNT(*) FROM team_projections_summary WHERE week = ?", (CURRENT_WEEK,))
    summaries = cursor.fetchone()[0]
    if summaries == EXPECTED_TEAMS:
        record_test("team_summary", "Team summaries", "PASS", f"{summaries} summaries")
    elif summaries > 0:
        record_test("team_summary", "Team summaries", "WARNING", f"{summaries}/{EXPECTED_TEAMS} summaries")
    else:
        record_test("team_summary", "Team summaries", "FAIL", "No summaries")
    
    cursor.execute("""
        SELECT team_name, owner, total_mu, combined_sigma
        FROM team_projections_summary WHERE week = ? ORDER BY total_mu DESC
    """, (CURRENT_WEEK,))
    print("   Team projections:")
    for i, row in enumerate(cursor.fetchall(), 1):
        print(f"     {i:2}. {row['team_name']} ({row['owner']}): μ={row['total_mu']:.1f}, σ={row['combined_sigma']:.2f}")
    
    conn.close()
except Exception as e:
    record_test("team_lineups", "Error", "FAIL", str(e))

👥 TEAM LINEUPS & SUMMARIES

✅ [PASS] Week 13 lineups: 108 entries (~108 expected)
✅ [PASS] All teams present: 12 teams
✅ [PASS] Team summaries: 12 summaries
   Team projections:
      1. Team 3 (amir812): μ=139.7, σ=27.23
      2. Team 8 (sahirsyed30): μ=130.0, σ=26.52
      3. Team 10 (monkeyman966699696): μ=127.4, σ=26.18
      4. Team 5 (TBK41): μ=122.0, σ=27.05
      5. Team 6 (Jibraan): μ=120.2, σ=26.67
      6. Team 9 (Bilal879): μ=117.9, σ=26.19
      7. Team 7 (mehdidrissi): μ=117.6, σ=26.89
      8. Team 11 (Ammady): μ=116.4, σ=26.36
      9. Team 12 (sfaizi24): μ=114.2, σ=26.48
     10. Team 1 (xavierking4): μ=114.0, σ=26.43
     11. Team 2 (asadrafique): μ=109.3, σ=26.74
     12. Team 4 (umarrahman30): μ=109.0, σ=26.52


---
## 3️⃣ Monte Carlo Database Tests (`montecarlo.db`)

In [6]:
print("="*70)
print("🎲 MONTE CARLO DATABASE TESTS")
print("="*70 + "\n")

try:
    conn = sqlite3.connect(DB_MONTECARLO)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    
    cursor.execute("SELECT * FROM simulation_runs WHERE week = ? ORDER BY created_at DESC LIMIT 1", (CURRENT_WEEK,))
    run = cursor.fetchone()
    if run:
        record_test("montecarlo", "Simulation run", "PASS", f"run_id={run['run_id'][:8]}..., n_sims={run['n_simulations']}")
    else:
        record_test("montecarlo", "Simulation run", "FAIL", "No simulation run")
    
    cursor.execute("SELECT COUNT(*) FROM monte_carlo_simulations WHERE week = ?", (CURRENT_WEEK,))
    sim_count = cursor.fetchone()[0]
    expected = EXPECTED_SIMS_PER_RUN * EXPECTED_TEAMS
    if sim_count >= expected * 0.9:
        record_test("montecarlo", "Simulation data", "PASS", f"{sim_count:,} records")
    elif sim_count > 0:
        record_test("montecarlo", "Simulation data", "WARNING", f"{sim_count:,}/{expected:,} records")
    else:
        record_test("montecarlo", "Simulation data", "FAIL", "No simulation data")
    
    cursor.execute("SELECT COUNT(DISTINCT team_id) FROM monte_carlo_simulations WHERE week = ?", (CURRENT_WEEK,))
    teams = cursor.fetchone()[0]
    if teams == EXPECTED_TEAMS:
        record_test("montecarlo", "All teams simulated", "PASS", f"{teams} teams")
    else:
        record_test("montecarlo", "All teams simulated", "WARNING", f"{teams}/{EXPECTED_TEAMS} teams")
    
    cursor.execute("""
        SELECT team_name, AVG(total_points) as avg_pts, MIN(total_points) as min_pts, MAX(total_points) as max_pts
        FROM monte_carlo_simulations WHERE week = ? GROUP BY team_id ORDER BY avg_pts DESC LIMIT 6
    """, (CURRENT_WEEK,))
    print("   Top 6 by avg simulated points:")
    for row in cursor.fetchall():
        print(f"     {row['team_name']}: avg={row['avg_pts']:.1f}, range=[{row['min_pts']:.1f}, {row['max_pts']:.1f}]")
    
    conn.close()
except Exception as e:
    record_test("montecarlo", "Error", "FAIL", str(e))

🎲 MONTE CARLO DATABASE TESTS

✅ [PASS] Simulation run: run_id=seed_173..., n_sims=50000
✅ [PASS] Simulation data: 600,000 records
✅ [PASS] All teams simulated: 12 teams
   Top 6 by avg simulated points:
     Team 3: avg=139.7, range=[66.3, 402.6]
     Team 8: avg=130.2, range=[50.4, 341.9]
     Team 10: avg=127.4, range=[54.6, 342.5]
     Team 5: avg=122.0, range=[50.0, 318.4]
     Team 6: avg=120.2, range=[53.0, 364.0]
     Team 9: avg=117.8, range=[50.4, 395.8]


---
## 4️⃣ Betting Odds Database Tests (`odds.db`)

In [7]:
print("="*70)
print("💰 BETTING ODDS DATABASE TESTS")
print("="*70 + "\n")

try:
    conn = sqlite3.connect(DB_ODDS)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    
    cursor.execute("SELECT COUNT(DISTINCT team_id) FROM betting_odds_team_ou WHERE week = ?", (CURRENT_WEEK,))
    team_ou = cursor.fetchone()[0]
    if team_ou == EXPECTED_TEAMS:
        record_test("odds", "Team O/U odds", "PASS", f"{team_ou} teams")
    elif team_ou > 0:
        record_test("odds", "Team O/U odds", "WARNING", f"{team_ou}/{EXPECTED_TEAMS} teams")
    else:
        record_test("odds", "Team O/U odds", "FAIL", "No team O/U odds")
    
    cursor.execute("SELECT COUNT(*) FROM betting_odds_matchup_ml WHERE week = ?", (CURRENT_WEEK,))
    ml = cursor.fetchone()[0]
    if ml >= 6:
        record_test("odds", "Matchup ML odds", "PASS", f"{ml} matchups")
    elif ml > 0:
        record_test("odds", "Matchup ML odds", "WARNING", f"{ml}/6 matchups")
    else:
        record_test("odds", "Matchup ML odds", "FAIL", "No matchup ML odds")
    
    cursor.execute("SELECT COUNT(DISTINCT team_id) FROM betting_odds_highest_scorer WHERE week = ?", (CURRENT_WEEK,))
    high = cursor.fetchone()[0]
    cursor.execute("SELECT COUNT(DISTINCT team_id) FROM betting_odds_lowest_scorer WHERE week = ?", (CURRENT_WEEK,))
    low = cursor.fetchone()[0]
    if high == EXPECTED_TEAMS and low == EXPECTED_TEAMS:
        record_test("odds", "High/Low scorer odds", "PASS", f"High:{high}, Low:{low}")
    elif high > 0 or low > 0:
        record_test("odds", "High/Low scorer odds", "WARNING", f"High:{high}/{EXPECTED_TEAMS}, Low:{low}/{EXPECTED_TEAMS}")
    else:
        record_test("odds", "High/Low scorer odds", "FAIL", "No high/low odds")
    
    cursor.execute("""
        SELECT SUM(probability) as total FROM betting_odds_highest_scorer 
        WHERE week = ? GROUP BY run_id ORDER BY created_at DESC LIMIT 1
    """, (CURRENT_WEEK,))
    result = cursor.fetchone()
    if result and result['total']:
        prob_sum = result['total']
        if 0.99 <= prob_sum <= 1.01:
            record_test("odds", "Probability sum", "PASS", f"Sum = {prob_sum:.4f}")
        else:
            record_test("odds", "Probability sum", "WARNING", f"Sum = {prob_sum:.4f} (expected ~1.0)")
    
    cursor.execute("""
        SELECT team_name, owner, line, over_odds, under_odds
        FROM betting_odds_team_ou WHERE week = ? ORDER BY line DESC LIMIT 5
    """, (CURRENT_WEEK,))
    print("   Sample O/U lines (top 5):")
    for row in cursor.fetchall():
        print(f"     {row['team_name']} ({row['owner']}): {row['line']:.1f} | O:{row['over_odds']} U:{row['under_odds']}")
    
    conn.close()
except Exception as e:
    record_test("odds", "Error", "FAIL", str(e))

💰 BETTING ODDS DATABASE TESTS

✅ [PASS] Team O/U odds: 12 teams
✅ [PASS] Matchup ML odds: 6 matchups
✅ [PASS] High/Low scorer odds: High:12, Low:12
✅ [PASS] Probability sum: Sum = 1.0000
   Sample O/U lines (top 5):
     Team 3 (amir812): 136.7 | O:+100 U:-100
     Team 8 (sahirsyed30): 127.2 | O:-100 U:+100
     Team 10 (monkeyman966699696): 124.5 | O:-100 U:-100
     Team 5 (TBK41): 118.6 | O:-100 U:-100
     Team 6 (Jibraan): 116.9 | O:-100 U:+100


---
## 5️⃣ Betting API Validation (Critical for Betting Page)

**These tests validate the exact data structure the Betting page expects.**

In [8]:
print("="*70)
print("🎰 BETTING API VALIDATION - Matchups Consistency")
print("="*70 + "\n")

try:
    conn_league = sqlite3.connect(DB_LEAGUE)
    conn_league.row_factory = sqlite3.Row
    cursor_league = conn_league.cursor()
    
    conn_odds = sqlite3.connect(DB_ODDS)
    conn_odds.row_factory = sqlite3.Row
    cursor_odds = conn_odds.cursor()
    
    cursor_league.execute("SELECT DISTINCT roster_id FROM rosters")
    league_roster_ids = {row[0] for row in cursor_league.fetchall()}
    print(f"📋 Roster IDs in league.db: {sorted(league_roster_ids)}")
    
    cursor_league.execute("""
        SELECT matchup_id_number, GROUP_CONCAT(roster_id) as teams
        FROM matchups WHERE week = ?
        GROUP BY matchup_id_number ORDER BY matchup_id_number
    """, (CURRENT_WEEK,))
    league_matchups = {}
    for row in cursor_league.fetchall():
        teams = sorted([int(t) for t in row['teams'].split(',')])
        league_matchups[row['matchup_id_number']] = tuple(teams)
    print(f"   League.db matchups: {league_matchups}")
    
    cursor_odds.execute("""
        SELECT team1_id, team2_id FROM betting_odds_matchup_ml
        WHERE week = ? ORDER BY team1_id
    """, (CURRENT_WEEK,))
    odds_matchups = set()
    for row in cursor_odds.fetchall():
        pair = tuple(sorted([row['team1_id'], row['team2_id']]))
        odds_matchups.add(pair)
    print(f"   Odds.db matchups: {sorted(odds_matchups)}")
    
    league_pairs = set(league_matchups.values())
    if league_pairs == odds_matchups:
        record_test("betting_api", "Matchups consistency", "PASS", f"All {len(league_pairs)} matchups match")
    else:
        only_league = league_pairs - odds_matchups
        only_odds = odds_matchups - league_pairs
        record_test("betting_api", "Matchups consistency", "FAIL", 
                    f"Matchup mismatch! League only: {only_league}, Odds only: {only_odds}")
    
    conn_league.close()
    conn_odds.close()
except Exception as e:
    record_test("betting_api", "Matchups consistency", "FAIL", str(e))

🎰 BETTING API VALIDATION - Matchups Consistency

📋 Roster IDs in league.db: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
   League.db matchups: {1: (5, 8), 2: (1, 4), 3: (6, 11), 4: (10, 12), 5: (2, 9), 6: (3, 7)}
   Odds.db matchups: [(1, 4), (2, 9), (3, 7), (5, 8), (6, 11), (10, 12)]
✅ [PASS] Matchups consistency: All 6 matchups match


In [9]:
print("\n" + "="*70)
print("🎰 BETTING API VALIDATION - No Duplicate Entries")
print("="*70 + "\n")

try:
    conn = sqlite3.connect(DB_ODDS)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    
    cursor.execute("SELECT team_id, COUNT(*) as cnt FROM betting_odds_team_ou WHERE week = ? GROUP BY team_id HAVING cnt > 1", (CURRENT_WEEK,))
    ou_dupes = cursor.fetchall()
    if not ou_dupes:
        cursor.execute("SELECT COUNT(DISTINCT team_id) FROM betting_odds_team_ou WHERE week = ?", (CURRENT_WEEK,))
        record_test("betting_api", "Team O/U no duplicates", "PASS", f"{cursor.fetchone()[0]} unique teams")
    else:
        record_test("betting_api", "Team O/U no duplicates", "FAIL", f"{len(ou_dupes)} teams have duplicates")
    
    cursor.execute("SELECT team_id, COUNT(*) as cnt FROM betting_odds_highest_scorer WHERE week = ? GROUP BY team_id HAVING cnt > 1", (CURRENT_WEEK,))
    hs_dupes = cursor.fetchall()
    if not hs_dupes:
        cursor.execute("SELECT COUNT(DISTINCT team_id) FROM betting_odds_highest_scorer WHERE week = ?", (CURRENT_WEEK,))
        record_test("betting_api", "Highest Scorer no duplicates", "PASS", f"{cursor.fetchone()[0]} unique teams")
    else:
        record_test("betting_api", "Highest Scorer no duplicates", "FAIL", f"{len(hs_dupes)} teams have duplicates")
    
    cursor.execute("SELECT team_id, COUNT(*) as cnt FROM betting_odds_lowest_scorer WHERE week = ? GROUP BY team_id HAVING cnt > 1", (CURRENT_WEEK,))
    ls_dupes = cursor.fetchall()
    if not ls_dupes:
        cursor.execute("SELECT COUNT(DISTINCT team_id) FROM betting_odds_lowest_scorer WHERE week = ?", (CURRENT_WEEK,))
        record_test("betting_api", "Lowest Scorer no duplicates", "PASS", f"{cursor.fetchone()[0]} unique teams")
    else:
        record_test("betting_api", "Lowest Scorer no duplicates", "FAIL", f"{len(ls_dupes)} teams have duplicates")
    
    cursor.execute("SELECT team1_id, team2_id, COUNT(*) as cnt FROM betting_odds_matchup_ml WHERE week = ? GROUP BY team1_id, team2_id HAVING cnt > 1", (CURRENT_WEEK,))
    ml_dupes = cursor.fetchall()
    if not ml_dupes:
        cursor.execute("SELECT COUNT(*) FROM betting_odds_matchup_ml WHERE week = ?", (CURRENT_WEEK,))
        record_test("betting_api", "Matchup ML no duplicates", "PASS", f"{cursor.fetchone()[0]} matchups")
    else:
        record_test("betting_api", "Matchup ML no duplicates", "FAIL", f"{len(ml_dupes)} duplicate matchups")
    
    conn.close()
except Exception as e:
    record_test("betting_api", "Duplicate check", "FAIL", str(e))


🎰 BETTING API VALIDATION - No Duplicate Entries

✅ [PASS] Team O/U no duplicates: 12 unique teams
✅ [PASS] Highest Scorer no duplicates: 12 unique teams
✅ [PASS] Lowest Scorer no duplicates: 12 unique teams
✅ [PASS] Matchup ML no duplicates: 6 matchups


In [10]:
print("\n" + "="*70)
print("🎰 BETTING API VALIDATION - All Teams Represented")
print("="*70 + "\n")

try:
    conn_league = sqlite3.connect(DB_LEAGUE)
    cursor_league = conn_league.cursor()
    conn_odds = sqlite3.connect(DB_ODDS)
    cursor_odds = conn_odds.cursor()
    
    cursor_league.execute("SELECT DISTINCT roster_id FROM rosters")
    roster_ids = {row[0] for row in cursor_league.fetchall()}
    
    cursor_odds.execute("SELECT DISTINCT team_id FROM betting_odds_team_ou WHERE week = ?", (CURRENT_WEEK,))
    ou_ids = {row[0] for row in cursor_odds.fetchall()}
    ou_missing = roster_ids - ou_ids
    if not ou_missing:
        record_test("betting_api", "Team O/U all teams", "PASS", f"All {len(roster_ids)} teams have O/U")
    else:
        record_test("betting_api", "Team O/U all teams", "FAIL", f"Missing teams: {ou_missing}")
    
    cursor_odds.execute("SELECT DISTINCT team_id FROM betting_odds_highest_scorer WHERE week = ?", (CURRENT_WEEK,))
    hs_ids = {row[0] for row in cursor_odds.fetchall()}
    hs_missing = roster_ids - hs_ids
    if not hs_missing:
        record_test("betting_api", "Highest Scorer all teams", "PASS", f"All {len(roster_ids)} teams have odds")
    else:
        record_test("betting_api", "Highest Scorer all teams", "FAIL", f"Missing teams: {hs_missing}")
    
    cursor_odds.execute("SELECT DISTINCT team_id FROM betting_odds_lowest_scorer WHERE week = ?", (CURRENT_WEEK,))
    ls_ids = {row[0] for row in cursor_odds.fetchall()}
    ls_missing = roster_ids - ls_ids
    if not ls_missing:
        record_test("betting_api", "Lowest Scorer all teams", "PASS", f"All {len(roster_ids)} teams have odds")
    else:
        record_test("betting_api", "Lowest Scorer all teams", "FAIL", f"Missing teams: {ls_missing}")
    
    cursor_odds.execute("SELECT team1_id, team2_id FROM betting_odds_matchup_ml WHERE week = ?", (CURRENT_WEEK,))
    ml_teams = set()
    for row in cursor_odds.fetchall():
        ml_teams.add(row[0])
        ml_teams.add(row[1])
    ml_missing = roster_ids - ml_teams
    if not ml_missing:
        record_test("betting_api", "Matchup ML all teams", "PASS", f"All {len(roster_ids)} teams in matchups")
    else:
        record_test("betting_api", "Matchup ML all teams", "FAIL", f"Missing teams: {ml_missing}")
    
    conn_league.close()
    conn_odds.close()
except Exception as e:
    record_test("betting_api", "Coverage check", "FAIL", str(e))


🎰 BETTING API VALIDATION - All Teams Represented

✅ [PASS] Team O/U all teams: All 12 teams have O/U
✅ [PASS] Highest Scorer all teams: All 12 teams have odds
✅ [PASS] Lowest Scorer all teams: All 12 teams have odds
✅ [PASS] Matchup ML all teams: All 12 teams in matchups


---
## 📊 Final Summary

In [11]:
print("\n" + "="*70)
print("📊 FINAL TEST SUMMARY")
print("="*70 + "\n")

passed = sum(1 for t in test_results if t['status'] == 'PASS')
warnings = sum(1 for t in test_results if t['status'] == 'WARNING')
failed = sum(1 for t in test_results if t['status'] == 'FAIL')
total = len(test_results)

print(f"Week {CURRENT_WEEK} Validation Results:")
print(f"  ✅ PASSED:   {passed}/{total}")
print(f"  ⚠️  WARNINGS: {warnings}/{total}")
print(f"  ❌ FAILED:   {failed}/{total}")
print()

categories = {}
for t in test_results:
    cat = t['category']
    if cat not in categories:
        categories[cat] = {'PASS': 0, 'WARNING': 0, 'FAIL': 0}
    categories[cat][t['status']] += 1

print("By category:")
for cat, stats in categories.items():
    print(f"  {cat}: ✅{stats['PASS']} ⚠️{stats['WARNING']} ❌{stats['FAIL']}")

if failed > 0:
    print("\n❌ FAILED TESTS:")
    for t in test_results:
        if t['status'] == 'FAIL':
            print(f"  [{t['category']}] {t['test']}: {t['message']}")

if warnings > 0:
    print("\n⚠️  WARNINGS:")
    for t in test_results:
        if t['status'] == 'WARNING':
            print(f"  [{t['category']}] {t['test']}: {t['message']}")

print("\n" + "="*70)
if failed == 0 and warnings == 0:
    print("🎉 ALL TESTS PASSED! Database ready for Betting page.")
elif failed == 0:
    print("✅ No critical failures. Review warnings.")
else:
    print("❌ FAILURES DETECTED. Fix before using Betting page.")
print("="*70)


📊 FINAL TEST SUMMARY

Week 13 Validation Results:
  ✅ PASSED:   31/32
  ⚠️  WARNINGS: 1/32
  ❌ FAILED:   0/32

By category:
  league.db: ✅5 ⚠️0 ❌0
  projections: ✅4 ⚠️1 ❌0
  sleeper_match: ✅1 ⚠️0 ❌0
  player_stats: ✅2 ⚠️0 ❌0
  team_lineups: ✅2 ⚠️0 ❌0
  team_summary: ✅1 ⚠️0 ❌0
  montecarlo: ✅3 ⚠️0 ❌0
  odds: ✅4 ⚠️0 ❌0
  betting_api: ✅9 ⚠️0 ❌0

⚠️  WARNINGS:
  [projections] Source coverage: Missing: {'firstdown.studio'}

✅ No critical failures. Review warnings.
